<a href="https://colab.research.google.com/github/AyaNabih7/Spam_Text_classication_roberta_base_model/blob/main/Spam_Text_classication_roberta_base_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Text Classification with roberta-base pre-trained model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from sklearn.datasets import load_files
import pandas as pd
SPAM= pd.read_csv(r"/content/drive/MyDrive/SPAM.csv")

In [ ]:
SPAM

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [ ]:
labels = pd.get_dummies(SPAM.Category, prefix='Category')
X = SPAM.iloc[:, -1].values
y = labels.values

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [ ]:
!pip install ktrain

STEP 1: Preprocess Data and Build a Transformer Model

In [ ]:
import ktrain
from ktrain import text
MODEL_NAME = 'roberta-base' #Pre-trained model
t = text.Transformer(MODEL_NAME, maxlen=500, class_names=categories)
trn = t.preprocess_train(X_train, y_train)
val = t.preprocess_test(X_test, y_test)
model = t.get_classifier()
learner = ktrain.get_learner(model, train_data=trn, val_data=val, batch_size=6)

preprocessing train...
language: en
train sequence lengths:
	mean : 16
	95percentile : 33
	99percentile : 53


Is Multi-Label? False
preprocessing test...
language: en
test sequence lengths:
	mean : 16
	95percentile : 33
	99percentile : 57


STEP 3: Train Model¶

In [ ]:
learner.fit_onecycle(8e-5, 1)



begin training using onecycle policy with max lr of 8e-05...
743/743 [==============================] - 548s 720ms/step - loss: 0.2061 - accuracy: 0.9401 - val_loss: 0.3943 - val_accuracy: 0.8664


STEP 4: Evaluate/Inspect Model

In [ ]:
learner.validate(class_names=t.get_classes())

              precision    recall  f1-score   support

         ham       0.87      1.00      0.93       966
        spam       0.00      0.00      0.00       149

    accuracy                           0.87      1115
   macro avg       0.43      0.50      0.46      1115
weighted avg       0.75      0.87      0.80      1115



/usr/local/lib/python3.7/dist-packages/sklearn/metrics/_classification.py:1221: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


array([[966,   0],
       [149,   0]])

STEP 5: Make Predictions on New Data in Deployment



In [ ]:
predictor = ktrain.get_predictor(learner.model, preproc=t)

In [ ]:
predictor.predict('FREE.')

'ham'

In [ ]:
! pip install git+https://github.com/amaiya/eli5@tfkeras_0_10_1

  Cloning https://github.com/amaiya/eli5 (to revision tfkeras_0_10_1) to /tmp/pip-req-build-nbkr7nz0
  Running command git clone -q https://github.com/amaiya/eli5 /tmp/pip-req-build-nbkr7nz0
  Running command git checkout -b tfkeras_0_10_1 --track origin/tfkeras_0_10_1
  Switched to a new branch 'tfkeras_0_10_1'
  Branch 'tfkeras_0_10_1' set up to track remote branch 'tfkeras_0_10_1' from 'origin'.
  Created wheel for eli5: filename=eli5-0.10.1-py2.py3-none-any.whl size=106850 sha256=9ffd1794b33e2ded08283f87da9d22d343896cd6aae809c697ad68548f10607c
  Stored in directory: /tmp/pip-ephem-wheel-cache-cjsj7rly/wheels/51/59/0a/0f48442b8d209583a4453580938d7ba2270aca40edacee6d45
Successfully built eli5


In [ ]:
predictor.explain('we can  use image reconstruction simulation bone.')